# Dust Coagulation Tutorial 1 - Simulation

In this tutorial, we will setup a dust coagulation simulation in order to generate training data for the neural network emulator that we will be training in the second part of the tutorial.

For the coagulation simulation, we will be using the [COALA code](https://github.com/mlombart/Coala), developed by [Lombart et al 2021](https://ui.adsabs.harvard.edu/abs/2021MNRAS.501.4298L/abstract), which provides a solver for the Smoluchowski coagulation equation.

Specifically, we will use a non-public python implementation of COALA, which you can find in the *coala_py* folder that is provided along with this notebook. For more details on the COALApy version, see also the *wiki_coala_py.pdf* in the *coala_py* folder. As we will be using python, we have prepared a conda environment with all the necessary python packages to run the COALApy implementation.

To install the environment, use your favourite conda environment handler (e.g. [Miniforge](https://github.com/conda-forge/miniforge)) and run the following in the terminal in the project folder:

`conda env create -f requirements_coala.yml`

After you have installed in the enviroment, you can activate it with:

`conda activate coala`

The environment comes with a jupyter-lab installation for your convenience to work on this notebook. The only other file needed for this first tutorial is the *wrapper_ai_coala_coag.py* python script, which provides a few prepared functions that we are going to need in the following.

## 1. Experiment with the coagulation simulator

In [ ]:
import os
import pandas
import numpy as np
import multiprocessing as mp

from coala_py.src import init_grid_log_phy
from wrapper_ai_coala_coag import init_power_law_rho_dust, compute_dv_ormel, wrapper_coala_coag_k0
from time import time

Below you will find a wrapper function for the dust coagulation simulation that we will use later for the training data generation. Given the initial conditions of gas density $\log(\rho_\mathrm{gas})$, dust to gas mass ratio log(D/G), turbulence parameter $\log(\alpha_\mathrm{turb})$, dust grain density $\rho_\mathrm{grain}$, and a specified number of timesteps $N_t$ to simulate, this routine will compute the initial $\log\rho_\mathrm{dust}^\mathrm{k}(t=0)$ and final dust grain size distributions $\log\rho_\mathrm{dust}^\mathrm{k}(\mathrm{dt})$ (with $k \in [1, \ldots, N_\mathrm{bins}])$ for all timesteps $\mathrm{dt}$ and return them in a single $N_t$ X M array. The input and output dust grain size distribution are returned in logspace for each bin.

In preparation for the training data generation, the routine will generate the timesteps uniformly random between 0 and `MAX_DT_IN_TFF`* $t_\mathrm{ff}$, where $t_\mathrm{ff}$ is the free fall time. For the purposes of this tutorial, we will model the dust grain size distribution using 100 size bins and up to $2 t_\mathrm{ff}$. However, you can vary the number of bins and the maximum simulation time by setting the `NBINS` and `MAX_DT_IN_TFF` global variables below. For simplicity, we will also keep the temperature `TEMP` fixed to $10\mathrm{K}$ for this tutorial. You may of course also experiment with this, when you are familiarising yourself with the simulation.

The additional global parameters `GRAV`, `SMAX`, `SMIN`, `SCUT`, `EPS_RHO_DUST` and `COEFF_PL` have already been set to reasonable values and do not need to be altered.

If you want to experiment with a different prescription for the timesteps, copy the setup from within the `run_coag_sim` routine below and use `wrapper_coala_coag_k0` to run the simulations.

In [ ]:
## Fixed global simulation parameters ##
TEMP = 10            # K,
NBINS = 100          # Number of grain size bins
MAX_DT_IN_TFF = 2.0  # Maximum simulation time as fraction of free-fall time

GRAV = 6.67e-8  #cgs
SMAX = 1e-1     #cgs
SMIN = 5e-7     #cgs
SCUT = 250e-7   #cgs
EPS_RHO_DUST = 1e-50
COEFF_PL = -3.5 #MRN distribution

# Wrapper function for dust coagulation simulation
def run_coag_sim(log_rho_gas: float,
                 log_dtg: float,
                 log_alpha_turb: float,
                 rhograin: float,
                 grainsize: List[float],
                 nsteps_dt: int,
                 bin_scut: int) -> np.array:
    '''
    Compute coagulation model for one set of initial conditions

    Parameters
    ----------
    log_rho_gas : float
        Log gas density in cgs.
    log_dtg : float
        Log dust to gas ratio.
    log_alpha_turb : float
        Log alpha turbulence.
    rhograin : float
        intrinsic grain density
    grainsize : float 
        
    nsteps_dt : int
        number of uniformly random sampled timesteps to draw between 0 and MAX_DT_IN_TFF
    bin_scut : int
        index of cut off for grainsize bins

    Returns
    -------
    output : np.array
        Array with initial conditions, dt, input density distribution and output density distribution

    '''

    output = np.zeros((nsteps_dt, 5 + NBINS * 2))

    verbose_coala = False
    rho_gas = 10**log_rho_gas
    dtg = 10**log_dtg
    alpha_turb = 10**log_alpha_turb

    tff = np.sqrt(3. * np.pi/(32. * GRAV * rho_gas))

    massgrid, massbins = init_grid_log_phy(NBINS, SMAX, SMIN, rhograin, 1.)

    # Compute the input dust grainsize distribution
    rho_dust_in = init_power_law_rho_dust(EPS_RHO_DUST,
                                          NBINS,
                                          massgrid,
                                          massbins,
                                          bin_scut,
                                          COEFF_PL,
                                          dtg,
                                          rho_gas)

    # Compute 2D array for grain-grain differential velocity from Ormel's model
    dv_ormel = compute_dv_ormel(rho_gas, TEMP, rhograin, grainsize, tff, alpha_turb)

    # Uniformly random sample time steps to compute grainsize distribution for
    dt_samples = np.random.uniform(0, MAX_DT_IN_TFF, nsteps_dt) * tff
    dt_samples.sort()

    # Store initial conditions, time steps and initial dust grain size distribution
    output[:, 0] = log_rho_gas
    output[:, 1] = log_dtg
    output[:, 2] = log_alpha_turb
    output[:, 3] = rhograin
    output[:, 4] = np.log10(dt_samples)
    output[:, 5:5+NBINS] = np.log10(rho_dust_in) 

    # Simulate dust grain size distributions for the sampled time steps
    wrapper_output = wrapper_coala_coag_k0(verbose_coala,
                                           EPS_RHO_DUST,
                                           rho_gas,
                                           rhograin,
                                           massgrid,
                                           massbins,
                                           dt_samples,
                                           dv_ormel,
                                           rho_dust_in)

    output[:, 5+NBINS:] = np.log10(wrapper_output)
    return output

The wrapper function above has been set up with the later incorporation into a training data generation script in mind. The input `bin_scut` is derived based on the number of bins to be simulated for the dust grain size distribution and does not depend on the initial conditions. As such, it can be precomputed to speed things up. You can compute `bin_scut` as follows: 

In [ ]:
size_grid = np.logspace(np.log10(SMIN), np.log10(SMAX), NBINS+1)
grainsize = np.sqrt(size_grid[1:] * size_grid[0:NBINS])

for j in range(NBINS):
    if (size_grid[j] < SCUT <= size_grid[j+1]):
        bin_scut = j

Now try to run the dust coagulation for a couple of different initial conditions. For reasonable limits to pick from see the description under 2. further below.

In [ ]:
# Simulation code goes here


### Visualise the simulations

Visualise the evolution of the dust grainsize distribution for a couple of different initial conditions. To compute the timestep size returned by the wrapper function as a fraction of the free fall time for the visualisation use

$t_\mathrm{ff} = \sqrt{\frac{3. \pi}{32G\rho_\mathrm{gas}}}$.

To get the grain sizes in $\mu m$ for each bin of the grain size distribution, you can use

In [ ]:
size_grid = np.logspace(np.log10(SMIN), np.log10(SMAX), NBINS+1)
grainsize = np.sqrt(size_grid[1:] * size_grid[0:NBINS])
cm_to_mu = 1e4
grainsize = grainsize * cm_to_mu

In [ ]:
# Visualisation code goes here


### Time the coagulation simulation
One of the primary reasons to look into machine learning based emulation of simulation codes is computation time. To see how much we can profit from the emulator, we therefore need a baseline for the execution time of dust coagulation simulation.

Use the `%timeit` function to get some statistics on the runtime of the `run_coag_sim` function, using 100 uniformly random sampled time steps.

In [ ]:
# Code to time wrapper function goes here


## 2. Setup a script to generate the training data

For the purpose of training the emulator, we now want to generate a large training data set with uniformly random sampled initial conditions. Specifically, we want to randomise the gas density $\rho_\mathrm{gas}$ (rho_gas), the dust to gas mass ratio D/G (dtg), the dust grain density $\rho_\mathrm{grain}$ (rhograin) and the turbulence parameter $\alpha_\mathrm{turb}$ (alpha_turb). For simplicity, we will keep the temperature fixed to $T = 10\mathrm{K}$.

**For $\rho_\mathrm{gas}$ consider a range of $10^{-20}$ to $10^{-11}\,\mathrm{g/cm}^{-3}$.**

**For D/G consider a range of $10^{-3}$ to $0.1$.**

**For $\alpha_\mathrm{turb}$ consider a range of $10^{-3}$ to $1.5$.**

**For $\rho_\mathrm{grain}$ consider a range of $1$ to $3\,\mathrm{g/cm}^{-1}$.**

Given the large dynamic range of $\rho_\mathrm{gas}$, D/G, and $\alpha_\mathrm{turb}$, it would be reasonable to sample these quantities in log space.

To make the training set generation feasible and a reasonable amount of time, parallelise the simulation for different initial conditions, using the `multiprocessing` package and the provided wrapper function for the coagulation simulation.

For this simple tutorial, we will generate a moderately sized training data set, consisting of **10000 initial conditions**, with **100** uniformly random sampled **time step sizes**. Save the output of the training data set generation in a `pandas.DataFrame` and save it as a pickle file with the `to_pickle` function of pandas dataframes.

In [ ]:
# Code for training data generation script goes here


In the second tutorial, we will try different setups for the emulation task. For that purpose, it will be helpful if we rearrange the generated training data in an alternative way.

As you have seen, the wrapper returns the simulation data in the following way:

$\left[\log(\rho_\mathrm{gas}), \log(D/G), \log(\alpha_\mathrm{turb}), \rho_\mathrm{grain}, \log(\mathrm{dt_i}), \log{\rho}_\mathrm{dust}^\mathrm{0}(t=0), \ldots, \log{\rho}_\mathrm{dust}^\mathrm{N_\mathrm{bins}}(t=0), \log{\rho}_{\mathrm{dust}}^\mathrm{0}(\mathrm{dt_i}), \ldots, \log{\rho}_{\mathrm{dust}}^\mathrm{N_\mathrm{bins}}(\mathrm{dt_i}) \right]$

In particular, $\log{\rho}_{\mathrm{dust}}^\mathrm{k}(t=0)$ is the same for all time steps $\mathrm{dt_i}$ that were generated for a given combination of initial conditions $\left[\log(\rho_\mathrm{gas}), \log(D/G), \log(\alpha_\mathrm{turb}), \rho_\mathrm{grain}\right]$.

That is to say, each row currently is a pair of the initial dust grain size distribution at $t=0$ and at time $\mathrm{dt_i}$. 

As an alternative, we will rearrange the training data set, such that we have pairs of the dust grain size distribution at time $t_{i-1}$ and $t_{i}$, such that each row reads for a given set of initial conditions:

$\left[\log(\rho_\mathrm{gas}), \log(D/G), \log(\alpha_\mathrm{turb}), \rho_\mathrm{grain}, \mathrm{\Delta t_i}, \log{\rho}_{\mathrm{dust}}^\mathrm{0}(\mathrm{dt_{i-1}}), \ldots, \log{\rho}_{\mathrm{dust}}^\mathrm{N_\mathrm{bins}}(\mathrm{dt_{i-1}}), \log{\rho}_{\mathrm{dust}}^\mathrm{k}(\mathrm{dt_i}), \ldots, \log{\rho}_{\mathrm{dust}}^\mathrm{N_\mathrm{bins}}(\mathrm{dt_i})\right]$

where $\Delta t_i = \mathrm{dt}_i - \mathrm{dt}_{i-1}$, and $\log{\rho}_{\mathrm{dust}}^\mathrm{k}(\mathrm{dt}_0) = \log{\rho}_{\mathrm{dust}}^\mathrm{k}(t=0)$

In [ ]:
# Code for rearranging the output goes here
